<a href="https://colab.research.google.com/github/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/Wiki_QA_v_2_SimpleApproach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Design  of  Simple 1 Step Wiki QA System

## 1 Step QA system (No Router,  Direct QA LLM call):

1. Single LLM call - Just let LLM choose the right topic and find answer
2. Just a simple Index (Table of Contents with simple summary/keyword for each topic)




*   Simple Index
https://github.com/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/WikiContents/WikiContentVersions/WikiSimpleVer3/SimpleIndex.md


In [1]:
import os
import shutil

if True==False:
  directory_to_delete = "/content/LLM_Wiki_Vault"

  # Check if the directory exists before attempting to delete
  if os.path.exists(directory_to_delete):
      shutil.rmtree(directory_to_delete)
      print(f"Directory '{directory_to_delete}' and its contents deleted successfully.")
  else:
      print(f"Directory '{directory_to_delete}' does not exist.")

In [2]:
# @title
OPENAI_KEY= '' # load from Colab Secrets




In [8]:
!mkdir -p /content/LLM_Wiki_Vault/wiki/


In [10]:
!unzip '/content/wiki.zip' -d /content/LLM_Wiki_Vault/

Archive:  /content/wiki.zip
replace /content/LLM_Wiki_Vault/wiki/anemia_during_pregnancy.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/LLM_Wiki_Vault/wiki/anemia_during_pregnancy.md  
  inflating: /content/LLM_Wiki_Vault/wiki/breastfeeding_guidance.md  
  inflating: /content/LLM_Wiki_Vault/wiki/breastfeeding_special_conditions.md  
  inflating: /content/LLM_Wiki_Vault/wiki/childbirth_and_newborn_care.md  
  inflating: /content/LLM_Wiki_Vault/wiki/child_growth_and_malnutrition.md  
  inflating: /content/LLM_Wiki_Vault/wiki/complementary_feeding_6_to_24_months.md  
  inflating: /content/LLM_Wiki_Vault/wiki/general_child_care_and_development.md  
  inflating: /content/LLM_Wiki_Vault/wiki/government_cash_benefit_schemes.md  
  inflating: /content/LLM_Wiki_Vault/wiki/government_free_service_schemes.md  
  inflating: /content/LLM_Wiki_Vault/wiki/immunization_rules_and_exceptions.md  
  inflating: /content/LLM_Wiki_Vault/wiki/immunization_schedule_and_basics.md  
  inflati

In [22]:
#  STEP 1:
# =====================================================================

import os
import json
from typing import List, Set
from pydantic import BaseModel, Field, field_validator, ValidationError
from google.colab import userdata


try:
    import openai
    from openai import types
except ImportError:
    print("  SDK missing from active runtime. Initializing pipeline pip deployment hooks...")
    !pip install -q openai
    import openai
    from openai import types


OPENAI_TOKEN_AUTHENTICATOR = OPENAI_KEY

openai_client = openai.OpenAI(api_key= OPENAI_KEY )
print("  authenticated.")



#  local path
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"



  authenticated.


In [23]:
# =====================================================================
# STEP 3  : SNEHA DIDI SYNTHESIS PROMPT for WIKI
# =====================================================================

SYSTEM_INSTRUCTION_SYNTHESIS = """

You are SNEHA DIDI. You’re a chatbot respond with to queries from women located in low economic  urban settlements on early childhood healthcare, pregnancy, government schemes and other related issues.
The women asking questions are not educated, will send messages with typos or incomplete information.
They need responses in language a 5 year old will understand, using simple and casual words, comprehensive, yet concise - a maximum  of 4-5 lines per response.


Rules of Responding:
1. If the question is unclear, ask 1-2 clarifying questions to better understand their needs/ gather more context.
For example, if the user asks any questions related to baby, ask questions like age of the baby, what they want to know etc before finding the answer.
2. What you share must be verifiable. So strictly limit your responses to what is there in the files in your knowledge base"
If there is a supporting video link, share that as well. Do not respond from the Internet or Hallucinate.
3.If you cannot find the answer in the documents, politely respond by saying 'I do not have enough information to answer your question'  in romanized hindi.
4. Maintain the style of messaging the user message you with.
5. If you get emojis/ generic greeting/ acknowledgment messages in any language(like yes, thank you, ok, ji, G, hi, bye, theek hai), respond according to their message.
6. You offer 'jaankari', not 'madad'

Language handling
1. User can send messages in Hindi (Devanagari script) or Romanised Hindi (English script with transliterated or translated hindi words).
If the message is in Devanagari script, strictly respond in Hindi using Devanagari script. If its in romanised Hindi, strictly use the same.
If the ask in English, strictly respond in romanised Hindi.
2. Avoid using the '!'  in your messages.
3.Do not use numbered lists in your response. Bullet point with un-numbered lists
4. Topics you do NOT respond to and 'I do not have enough information to answer your question' - children aged 1, pregnancy sex questions, family planning
5. Use the hindi word for baby, iron (the metal in our blood stream)
6. Use simpler words than unclear, specific, cost, growth, meals, healthy, tummy, facilities, seasonal, hydrated, bleeding, fever, guava, organs, structure, placenta, junk, variety, mashed, quantity, soft, mackerel, salmon, mercury, absorb, legumes, citrus - use colloquial words the women can understand.

"""



print("✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.")


✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.


In [25]:


# --- STAGE 3 QA SYNTHESIS   INSTRUCTIONS





TEMPLATE_SYNTHESIS = """
Your goal is to answer the Incoming EndUser Question by extracting answer from the provided Context

Your Task: Answer the EndUser Question using the provided Context.
Suggested Steps to Answer the Question:
1. Identify the relevent topic(s) of the question by looking up with Index File ( The index file identifier is index_stage_routing_protocol.md in the enclosed context)
2. Extract answers from the identified relevent topicX.md files.
3. Adhere to Rules of Responding such as provided 100% grounded answers from the provided Content.

Your inputs:
1. Incoming EndUser Query (EndUser's Question) .
     Incoming End User Question: "{user_whatsapp_query}"

2. The context to answer from: The  wiki context blocks organized by topics (You are given seperate Wiki File for each topic)

      You are given a  {howmany}  files. This includes a single Index file and many topic files.
      The contents of these {howmany} files are presented below one after another, starting with the Index file.
      The 1st file in the below Wiki Files is the Index file, identifed as
      These {howmany} Wiki files are as presented in the following sequence.

      Table of Contents of Files:
      {tableofcontentoffiles}

      Below are {howmany} markdown files.

      Wiki Content to extract the answer from:
      ==================================================
      ===  Wiki Files ===
      {packaged_context_string_payload}
      ==================================================


"""




In [26]:
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

def get_allowed_topic_files_manifest(wiki_dir: str) -> Set[str]:
    """
      scans the   folder path   and populates a strict local memory hash-set of valid, available topic files.
    """
    if not os.path.exists(wiki_dir):
        print(f"⚠️  Directory Alert: Target vault path '{wiki_dir}' does not exist on disk yet. "
              f"Please ensure you drop your exported 'wiki/' folder inside the Colab file storage directory.")
        return set()

    # Use a set to automatically handle duplicates
    all_files_found_set = set()

    # Add all .md files found in the directory to the set
    for f in os.listdir(wiki_dir):
        if f.endswith(".md"):
            all_files_found_set.add(f)

    # Convert the set to a list
    all_files_found = list(all_files_found_set)

    # Ensure 'index_stage_routing_protocol.md' is the first element if it exists
    if 'index_stage_routing_protocol.md' in all_files_found:
        all_files_found.remove('index_stage_routing_protocol.md')
        all_files_found.insert(0, 'index_stage_routing_protocol.md')

    return all_files_found

ALLOWED_WIKI_FILES = get_allowed_topic_files_manifest(WIKI_VAULT_DIR)
print(f" [Step 1 Status]: Harvested allowed local topic files footprint: {ALLOWED_WIKI_FILES}")

 [Step 1 Status]: Harvested allowed local topic files footprint: ['index_stage_routing_protocol.md', 'pregnancy_nutrition_and_diet.md', 'pregnancy_care_and_checkups.md', 'local_health_services_and_schedules.md', 'anemia_during_pregnancy.md', 'immunization_rules_and_exceptions.md', 'complementary_feeding_6_to_24_months.md', 'child_growth_and_malnutrition.md', 'vaccine_side_effects_and_reactions.md', 'breastfeeding_special_conditions.md', 'immunization_schedule_and_basics.md', 'government_cash_benefit_schemes.md', 'government_free_service_schemes.md', 'breastfeeding_guidance.md', 'general_child_care_and_development.md', 'childbirth_and_newborn_care.md']


In [40]:
def execute_downstream_qa_synthesis(
    openai_client_instance,
    wiki_dir: str,
    user_query: str,
    routing_payload_dict: dict,
    target_gpt_model: str = "gpt-4o-mini"
) -> str:
    """
    Reads selected files from disk, appends context, and generates a context-bounded answer.
    Dynamically adjusts API arguments to accommodate differences between gpt-4o-mini and gpt-5.6-terra.
    """
    # STEP A:
    target_slugs = ALLOWED_WIKI_FILES

   # print("DEBUG router output")
   # print(routing_payload_dict)
   # print("DEBUG router output")

    #if not target_slugs:
     #   print("⏭️  [Pipeline Short-Circuit]: Router returned zero topic matches. Aborting Stage 3 execution.")
      #  return "I am sorry, but that query falls completely outside the scope of our authoritative field manual guidelines."

    # =================================================================
    # load the selected files (these files were shortlisted by router in stage 1 )
    # =================================================================
    print(f"[Stage 2 Injector] 📁 Router selected {len(target_slugs)} targets. Loading files from disk...")
    packaged_context_chunks = []

    #print('DEBUG List of Identified files')
    #print(target_slugs)
    #print('DEBUG End of list of Identified files')

    blocknumber = 0
    tableofcontentoffiles = ""

    for filename in target_slugs:
        blocknumber = blocknumber + 1
        clean_name = filename.strip()
        full_file_path = os.path.join(wiki_dir, clean_name)

        if not os.path.exists(full_file_path):
            print(f"    ⚠️  Warning: Router referenced file '{clean_name}', but it is missing from disk path '{wiki_dir}'. Skipping.")
            continue

        # Open file and read
        with open(full_file_path, 'r', encoding='utf-8') as f_in:
            raw_file_text = f_in.read()

        tableofcontentoffiles = tableofcontentoffiles + f" {blocknumber}: {clean_name}\n"
        # Structure the chunk part inside explicit visual code containment boundaries
        bounded_chunk_part = (
            f"--------------------------------------------------\n"
            f"<Wiki_Context_Block > \n"
            f" File Number {blocknumber} : Unique Wiki Filename: {clean_name} \n"
            f"--- START COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"{raw_file_text}\n"
            f"--- END COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"</Wiki_Context_Block> \n"
            f"--------------------------------------------------\n"
        )
        packaged_context_chunks.append(bounded_chunk_part)

    # Unify all selected file text segments into a single context payload block
    unified_context_string = "\n".join(packaged_context_chunks)
    print(f"[Stage 2 Injector] ✅ Context payload compiled. Total text size: {len(unified_context_string)} characters.")

    # =================================================================
    # STAGE 3: THE GPT content SYNTHESIS - send context files to gpt
    # =================================================================
    print(f"[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via {target_gpt_model}...")

    # Template the unified context string straight into the user prompt payload
    user_prompt_string = TEMPLATE_SYNTHESIS.format(
        packaged_context_string_payload=unified_context_string,
        user_whatsapp_query=user_query,
        howmany=len(target_slugs),
        listoffilenames=target_slugs,
        tableofcontentoffiles = tableofcontentoffiles,

    )

    # Determine system role: GPT-5 tier requires 'developer' role for systemic instructions
    system_role = "developer" if "gpt-5.6" in target_gpt_model else "system"

    messages_payload = [
        {"role": system_role, "content": SYSTEM_INSTRUCTION_SYNTHESIS},
        {"role": "user", "content": user_prompt_string}
    ]


    # Dynamically assemble the parameter keywords dict
    api_kwargs = {
        "model": target_gpt_model,
        "messages": messages_payload,
        "seed": 42,
        "temperature": 0 # Default to 0 for deterministic output
    }

    # Apply thinking effort exclusively to the GPT-5.6 reasoning flag tracking
    if "gpt-5" in target_gpt_model:
        api_kwargs["reasoning_effort"] = "medium"
        # Remove temperature for GPT-5.6 models if not supported
        if "temperature" in api_kwargs:
            del api_kwargs["temperature"]


   # print(api_kwargs)
    # Call the completions API
    response = openai_client_instance.chat.completions.create(**api_kwargs)

    if not response.choices:
        raise ValueError("❌ Stage 3 Synthesis Fault: OpenAI returned an empty choices list array block.")

    final_conversational_answer = response.choices[0].message.content.strip()
    print("[Stage 3 Synthesis] ✅ Airtight response generated successfully.")
    return final_conversational_answer

In [41]:
# =====================================================================
#  END-TO-END PIPELINE testing here
# =====================================================================

# Ensure WIKI_VAULT_DIR matches your local environment variable path precisely
# This folder must contain index_detailed.md and your compiled topic pages
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

TARGET_LLM_MODEL = "gpt-5.6-luna" # "gpt-4o-mini", "gpt-5.4-mini",  "gpt-5.6-luna"


# Define a high-stakes test query
TEST_USER_QUERY = "baby solid food recommned"


    # Run Step 2 & Load file and
    #        Step 3 QA Synthesis
chatbot_final_answer = execute_downstream_qa_synthesis(
        openai_client_instance=openai_client,
        wiki_dir=WIKI_VAULT_DIR,
        user_query=TEST_USER_QUERY,
        routing_payload_dict=None,
        target_gpt_model = TARGET_LLM_MODEL
    )

print("\n📱 ===================== WHATSAPP CHATBOT TEXT OUTPUT =====================")
print(chatbot_final_answer)



[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.

📱 ===================== WHATSAPP CHATBOT TEXT OUTPUT =====================
Aapka bachcha kitne mahine ka hai?  
6 mahine poore hone ke baad hi solid khana shuru karein.  
Shuru mein 2–3 chamach gaadha khichdi, dal-chawal, mashed kela ya paka hua aloo dein.  
Breastfeeding jaari rakhein.


In [42]:
 # 📱   GLIFFIC CHATBOT   INTERFACE
# =====================================================================



def get_chatbot_response(user_question: str) -> str:
    """
    Unified entry point for the downstream WhatsApp Chatbot pipeline.

    Input Parameter:
        user_question (str): The raw text message question received from the user.

    Output Return:
        str: The context-locked, sanitised, and cited answer text block.
    """
    # Assumes WIKI_VAULT_DIR and openai_client have been successfully
    WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

    try:
        final_answer_payload = execute_downstream_qa_synthesis(
            openai_client_instance=openai_client,
            wiki_dir=WIKI_VAULT_DIR,
            user_query=user_question,
            routing_payload_dict=None,
            target_gpt_model=TARGET_LLM_MODEL
        )

        # Return the pure, conversational string text block directly back to your interface router wrapper
        return final_answer_payload ,  None

    except Exception as runtime_pipeline_fault:
        # Operational System Catch Gate: Log error context inside your terminal execution log lines
        print(f"🚨 [Pipeline Fatal Anomaly]: Execution failed. Reason: {str(runtime_pipeline_fault)}")

        # Return a safe, defensive fallback error message token back to the WhatsApp platform queue
        return None, None


In [43]:
# unit test
finalanswer_to_giveto_user = get_chatbot_response("  3 month child  what to give")
print(finalanswer_to_giveto_user)


[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('3 mahine ke baby ko sirf maa ka doodh dein, baar-baar jab baby maange.  \nPaani, shahad, ghutti, juice ya koi aur khana na dein.  \nMaa ka doodh 6 mahine tak baby ke liye kaafi hota hai.  \n6 mahine poore hone ke baad hi halka ghar ka khana shuru karein.', None)


In [44]:
finalanswer_to_giveto_user = get_chatbot_response("  baby  diarrohia")
print(finalanswer_to_giveto_user)

[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Baby ki umar kitni hai? Dast kitni baar ho rahe hain, aur kab se?  \nBaby doodh/paani pee raha hai ya bahut sust hai? Bukhar, ulti, ya saans tez to nahi?  \nAgar baby doodh-paani nahi pee raha, bahut sust hai, tez saans ya zyada bukhar hai, to turant health worker ko dikhayein.  \nDoodh pilana jaari rakhein.', None)


In [45]:
finalanswer_to_giveto_user = get_chatbot_response("  there is cyclone alert here.   how to  induce pain in 9th for early delivery ")
print(finalanswer_to_giveto_user)



[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Cyclone ke darr se khud se dard lane ki koshish bilkul mat karo. 39 hafte tak bachche ko pet me rehne dena chahiye, jab tak doctor kuch aur na bole.  \nAbhi apne doctor ya najdeek ke sarkari aspataal se baat karo. Zarurat ho to 102/108 ambulance bulao.  \nKhoon, pani aana, tez pet dard, ya bachche ki harkat kam ho to turant aspataal jao. Aap kitne hafte ki pregnant ho?', None)


In [46]:
finalanswer_to_giveto_user = get_chatbot_response(" Is wakt kya kru ges khatm karne ke liye ")
print(finalanswer_to_giveto_user)

[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Is sawal ka jawab dene ke liye mere paas paryapt jaankari nahi hai.', None)


In [47]:
finalanswer_to_giveto_user = get_chatbot_response(" Dast padhne per kya khayen ")
print(finalanswer_to_giveto_user)

[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Dast kis ko ho rahe hain, bacche ko ya bade ko?  \nAgar bacche ko hain, to bacche ki umar kitni hai?  \nUmar batayen, phir main khane ke baare mein sahi jaankari dungi.', None)


In [48]:
get_chatbot_response(" folic tablet how many in 2nd month? ")


[Stage 2 Injector] 📁 Router selected 16 targets. Loading files from disk...
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 506511 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.


('2nd month mein folic acid **500 mcg (0.5 mg) ki 1 tablet roz** leni hoti hai.  \nFolic acid pehle 3 mahine mein bahut zaroori hai.  \nTablet ka naam aur taakat dekhkar hi lein, aur doctor/ANM ko dikha dein.',
 None)